In [11]:
# 1. 加载数据
import anndata as ad
import numpy as np

RNA_PATH = "/home/lihx/fm/bacdiff/data/OSCC_15.GEX.h5ad"
BAC_PATH = "/home/lihx/fm/bacdiff/data/OSCC_15.16S.h5ad"
OUT_PATH = "/home/lihx/fm/bacdiff/data/OSCC_15_merged.h5ad"

rna = ad.read_h5ad(RNA_PATH)
bac = ad.read_h5ad(BAC_PATH)

# 按细胞名对齐
common = sorted(set(rna.obs_names) & set(bac.obs_names))
rna = rna[common].copy()
bac = bac[common].copy()

bac_raw = bac.X.toarray() if hasattr(bac.X, 'toarray') else np.array(bac.X)
bac_names = list(bac.var_names)

print(f"RNA: {rna.n_obs} cells x {rna.n_vars} genes")
print(f"BAC: {bac.n_obs} cells x {bac.n_vars} bacteria")

RNA: 6469 cells x 38606 genes
BAC: 6469 cells x 29 bacteria


In [13]:
# 2. 参数：某种细菌在细胞中检出数低于此阈值，则剔除该细菌（整列删除）
min_cells_per_bacteria = 10 # 默认 100（至少在 100 个细胞中检出的细菌才保留）

cells_per_bacteria = (bac_raw > 0).sum(axis=0)
keep = cells_per_bacteria >= min_cells_per_bacteria

print(f"细菌过滤: {bac_raw.shape[1]} 种 -> {keep.sum()} 种 (保留检出 >= {min_cells_per_bacteria} cells 的)")

# 显示保留和被过滤的
for name, n_cells, k in zip(bac_names, cells_per_bacteria, keep):
    status = "保留" if k else "过滤"
    print(f"  [{status}] {name:30s} ({n_cells} cells)")

bac_raw = bac_raw[:, keep]
bac_names = [n for n, k in zip(bac_names, keep) if k]

细菌过滤: 3 种 -> 3 种 (保留检出 >= 10 cells 的)
  [保留] Ihuprevotella                  (12 cells)
  [保留] Patulibacter                   (57 cells)
  [保留] unclassified_Pseudomonas       (17 cells)


In [14]:
# 3. 细菌标签（用完整细菌名）
labels = []
for i in range(bac_raw.shape[0]):
    present = [bac_names[j] for j in range(len(bac_names)) if bac_raw[i, j] > 0]
    labels.append("_".join(present) if present else "none")

rna.obs["bacteria_label"] = labels

# 打印标签分布
from collections import Counter
print(f"标签种类: {len(set(labels))}")
for label, count in Counter(labels).most_common():
    print(f"  {label:60s} {count:5d}")

标签种类: 5
  none                                                          6384
  Patulibacter                                                    56
  unclassified_Pseudomonas                                        16
  Ihuprevotella                                                   12
  Patulibacter_unclassified_Pseudomonas                            1


In [15]:
# 4. 硬筛选：标签细胞数低于阈值的直接剔除
min_cells_per_label = 10  # 默认 100（该标签至少 100 个细胞才保留）

label_counts = rna.obs["bacteria_label"].value_counts()
keep_mask = rna.obs["bacteria_label"].isin(label_counts[label_counts >= min_cells_per_label].index)

print(f"标签过滤: {rna.n_obs} cells -> {keep_mask.sum()} cells (保留标签 >= {min_cells_per_label} cells)")
print(f"保留的标签:")
for label, count in label_counts[label_counts >= min_cells_per_label].items():
    print(f"  {label:60s} {count:5d}")
print(f"\n过滤掉的标签:")
for label, count in label_counts[label_counts < min_cells_per_label].items():
    print(f"  {label:60s} {count:5d}")

rna = rna[keep_mask].copy()
bac_raw = bac_raw[keep_mask.values]

标签过滤: 6469 cells -> 6468 cells (保留标签 >= 10 cells)
保留的标签:
  none                                                          6384
  Patulibacter                                                    56
  unclassified_Pseudomonas                                        16
  Ihuprevotella                                                   12

过滤掉的标签:
  Patulibacter_unclassified_Pseudomonas                            1


In [16]:
# 5. 合并 16S + GEX 写入新 h5ad
import pandas as pd

# 细菌浓度：原始 16S 值，存到 obsm
rna.obsm["bacteria_raw"] = bac_raw.astype(np.float32)

# 细菌总丰度
rna.obs["bacteria_total_raw"] = bac_raw.sum(axis=1).astype(np.float32)

# 细菌名存到 uns
rna.uns["bacteria_names"] = bac_names

rna.write_h5ad(OUT_PATH)
print(f"已保存: {OUT_PATH}")
print(f"Shape: {rna.n_obs} cells x {rna.n_vars} genes")
print(f"Bacteria: {len(bac_names)} 种")
print(f"Labels: {rna.obs['bacteria_label'].nunique()} 种")
print(f"Obs columns: {list(rna.obs.columns)}")
print(f"Obsm keys: {list(rna.obsm.keys())}")

已保存: /home/lihx/fm/bacdiff/data/OSCC_15_merged.h5ad
Shape: 6468 cells x 38606 genes
Bacteria: 3 种
Labels: 4 种
Obs columns: ['bacteria_label', 'bacteria_total_raw']
Obsm keys: ['bacteria_raw']


In [17]:
import scanpy as sc

adata1 = sc.read_h5ad("/home/lihx/fm/bacdiff/data/OSCC_11_merged.h5ad")

print(adata1)

/home/lihx/anaconda3/envs/concert/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/lihx/anaconda3/envs/concert/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/lihx/anaconda3/envs/concert/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/lihx/anaconda3/envs/concert/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/lihx/anaconda3/envs/concert/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing 

AnnData object with n_obs × n_vars = 5309 × 38606
    obs: 'bacteria_label', 'bacteria_total_raw'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'bacteria_names'
    obsm: 'bacteria_raw'


In [10]:
adata1.obs["bacteria_label"].value_counts().head(100)

bacteria_label
none                                                           2612
unclassified_Fusobacterium                                      985
Treponema                                                       604
Treponema_unclassified_Fusobacterium                            387
Treponema_unclassified_Treponema                                199
Treponema_unclassified_Fusobacterium_unclassified_Treponema     170
Fusobacterium_unclassified_Fusobacterium                        135
Fusobacterium_Treponema_unclassified_Fusobacterium              109
Fusobacterium                                                   108
Name: count, dtype: int64

In [8]:
adata1.obs['bacteria_total_raw'].value_counts().head(10)

bacteria_total_raw
0.0    2612
1.0    1194
2.0     425
3.0     224
4.0     133
5.0      89
8.0      69
6.0      66
7.0      52
9.0      38
Name: count, dtype: int64